# TabFM, tested honestly, the companion notebook

Google's TabFM predicts on tables it has never seen, no training, no tuning, one forward pass. In this notebook you run the exact experiments the episode verified, and then point them at your own data.

Quick honesty first, zero-shot tabular is not brand new. TabPFN pioneered it and TabICL pushed it further, TabFM is Google's hybrid of the two at Google scale. What you are testing here is whether it earns its size.

Watch the episode, [EPISODE VIDEO LINK]

Code, skill, and this notebook live together in the [topic folder](https://github.com/SaschaHeyer/gen-ai-livestream/tree/main/tabfm).

## 1. Setup

Everything installs from GitHub, pinned to the exact commit the episode verified. Two traps make this install line what it is, the PyPI package cannot load the published weights (its loader wants a file the release does not contain), and `safetensors` must be listed explicitly because tabfm needs it but does not declare it.

In [1]:
# 2 to 4 minutes on Colab. The 6.6 GB model weights download later, at the first load() call,
# about 6 minutes cold on the episode machine.
%pip install -q "tabfm[pytorch] @ git+https://github.com/google-research/tabfm@633cd265f498e1d20c9625be0639f6305d8e2541" safetensors scikit-learn pandas certifi

/private/tmp/claude-501/-Users-sascha-Desktop-development-backstage/ef4afa22-ed11-43e9-b521-44116aa61915/scratchpad/tabfm-prep/.venv-nb/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


Next cell, two guards before anything runs. TabFM refuses Python older than 3.11, and the default Hugging Face download backend (Xet) crashed mid-download during the episode's verification, plain HTTP is reliable.

> **Sharp edge.** Without `HF_HUB_DISABLE_XET=1` the 6.6 GB weight download can die halfway with an internal writer error. Set it before the first `load()`.

In [2]:
import os, sys

assert sys.version_info >= (3, 11), f"TabFM needs Python 3.11 or newer, this kernel is {sys.version.split()[0]}"
os.environ["HF_HUB_DISABLE_XET"] = "1"   # the reliable download path
print(f"python {sys.version.split()[0]}, download backend set, ready")

python 3.12.8, download backend set, ready


## 2. The hero, zero-shot classification in six lines

Imagine you hire a new analyst, and instead of studying your data for a week they glance at your spreadsheet once and start predicting. That glance and go is what zero-shot tabular means, the model is shown your training rows as context at inference time, nothing is ever trained.

We use the classic wine dataset, 178 rows, 13 features, 3 classes. Not because it is hard, because it fits TabFM's guardrails comfortably and everyone can read it. First the imports.

In [3]:
import time
import torch
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tabfm.src.pytorch import tabfm_v1_0_0
from tabfm import TabFMClassifier

Load the table and split it the boring honest way, stratified, fixed seed, 30 percent held out.

In [4]:
SEED = 42

X, y = load_wine(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)
print(f"{X.shape[0]} rows, {X.shape[1]} features, {y.nunique()} classes, "
      f"{len(y_train)} train / {len(y_test)} test")

178 rows, 13 features, 3 classes, 124 train / 54 test


Now the model. The first call downloads the 6.6 GB classification checkpoint from Hugging Face, later calls hit the cache.

> **Sharp edge.** On CPU pass `dtype=torch.float32`. The loader defaults to bfloat16, which ran about 4.5x slower on CPU in the episode's testing (220s vs 49s for the same predict). Keep the bfloat16 default on a GPU.

In [5]:
t0 = time.time()
model = tabfm_v1_0_0.load(dtype=torch.float32)
print(f"model load, {time.time()-t0:.1f}s")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory


model load, 11.6s


Fit and predict. Watch what fit does NOT do, there is no training loop, it only stashes your table as context. Predict is the single forward pass the whole pitch is about.

In [6]:
clf = TabFMClassifier(model=model)

t0 = time.time()
clf.fit(X_train, y_train)        # no training, stashes the table as context
preds = clf.predict(X_test)      # one forward pass
print(f"fit + predict, {time.time()-t0:.1f}s")

fit + predict, 37.0s


> **Sharp edge.** `predict()` returns the original labels in an object dtype array and sklearn metrics refuse the mix, cast before scoring.

In [7]:
acc = accuracy_score(y_test, preds.astype(int))
print(f"TabFM zero-shot accuracy, {acc:.4f}")

TabFM zero-shot accuracy, 1.0000


What to notice. No training bar crawled by, fit only stashed the table, that is the whole trick. The episode measured 49.2s for this on CPU and a perfect 1.0000, your numbers should land close.

### Your turn

Does the perfect score survive a different split. Change `SEED` at the top of the split cell to any number, rerun that cell and the two cells after it, and watch whether accuracy moves. One split is one draw.

In [8]:
# YOUR TURN, edit and rerun the cells above
# 1. set SEED = 7 (or any number) in the split cell
# 2. rerun split, fit + predict, accuracy
# a worked answer sits in the Solutions section at the bottom
print("edit SEED in the split cell above, then rerun the three cells that follow it")

edit SEED in the split cell above, then rerun the three cells that follow it


## 3. The honest benchmark, on data that is NOT wine

A perfect score means nothing on its own, the only question that matters is what the boring classics score on the same split. The episode ships `benchmark.py` for exactly this, it runs TabFM, XGBoost, and TabICL on any CSV, same split, and prints accuracy, time, and context side by side.

It also runs every model in its own subprocess on purpose, XGBoost and PyTorch bundle conflicting OpenMP runtimes on macOS and one process segfaults or deadlocks at 0 percent CPU.

Colab does not ship sibling files, so the next cell fetches the script from the topic folder, about 9 KB.

In [9]:
import ssl, urllib.request, certifi

url = "https://raw.githubusercontent.com/SaschaHeyer/gen-ai-livestream/main/tabfm/skill/scripts/benchmark.py"
ctx = ssl.create_default_context(cafile=certifi.where())   # explicit CA bundle, works on any Python build
with urllib.request.urlopen(url, context=ctx) as r, open("benchmark.py", "wb") as f:
    f.write(r.read())
print("fetched benchmark.py, ~9 KB")

fetched benchmark.py, ~9 KB


The test data is breast cancer, 569 rows, 30 features, 2 classes. Chosen because it is NOT wine, on wine all three models tie at 1.0000 and you learn nothing about their differences.

In [10]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

Xbc, ybc = load_breast_cancer(return_X_y=True, as_frame=True)
d = Xbc.copy(); d["diagnosis"] = ybc
d.to_csv("breast_cancer.csv", index=False)
print(f"breast_cancer.csv written, {d.shape[0]} rows, {d.shape[1]-1} features")

breast_cancer.csv written, 569 rows, 30 features


Now the race. The extra installs are for the two competitor models, then the benchmark runs all three. TabFM's part takes a few minutes on CPU, the printout tells you who is running.

In [11]:
%pip install -q xgboost tabicl

/private/tmp/claude-501/-Users-sascha-Desktop-development-backstage/ef4afa22-ed11-43e9-b521-44116aa61915/scratchpad/tabfm-prep/.venv-nb/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [12]:
import sys
!{sys.executable} benchmark.py breast_cancer.csv --target diagnosis

dataset: breast_cancer.csv, 569 rows, 30 features, 2 classes
running xgboost in its own process...


running tabicl in its own process...


running tabfm in its own process...



model       accuracy   seconds   context
xgboost       0.9532       0.6       398
tabicl        0.9591       3.7       398
tabfm         0.9766     157.3       398

A tie means the smaller model already does the job. This split is one draw, rerun with another --seed before deciding.


What to notice, and read both halves out loud. On the episode machine this was the first dataset where the three models separated, TabFM ahead on accuracy, 0.9766 against 0.9591 and 0.9532, and paying roughly 170 seconds for what XGBoost does in 0.2. Whether that trade is worth it is not a leaderboard question, it is a question about your table and your latency budget.

### Your turn

Point it at YOUR data. Any CSV with a label column of at most 10 classes works, the guardrails check the limits before the 6.6 GB model is touched.

In [13]:
# YOUR TURN, benchmark your own table
# in Colab, uncomment the two upload lines, pick a CSV, then run the benchmark line
# from google.colab import files
# uploaded = files.upload()
# !{sys.executable} benchmark.py YOUR_FILE.csv --target YOUR_LABEL_COLUMN
print("upload a CSV and point benchmark.py at it, guardrails included")

upload a CSV and point benchmark.py at it, guardrails included


## 4. The edge lab, prove the 10-class wall on purpose

The model card says classification caps at 10 classes. The episode did not take the card's word for it, and neither should you, the next cell builds an 11-class problem and runs into the wall deliberately. A reader who reproduces a limit on purpose has learned it for good.

In [14]:
import numpy as np

rng = np.random.default_rng(0)
X11 = rng.normal(size=(220, 5))
y11 = np.arange(220) % 11          # 11 classes, one past the cap

try:
    TabFMClassifier(model=model).fit(X11, y11)
    print("unexpected, 11 classes fit without error")
except ValueError as e:
    print(f"refused, {e}")

refused, The number of classes (11) exceeds the maximum number of classes (10) supported by the model.


What to notice, the refusal is immediate and names the limit, no compute was wasted. `benchmark.py` checks the same thing even earlier, before the model loads at all.

### Your turn

Make it pass. Change the label line so exactly 10 classes remain and rerun, the fit should go through. The model is already loaded in this kernel, so it costs seconds, not a download.

In [15]:
# YOUR TURN, drop one class so it fits
# hint, the modulo is the class count
print("change y11 so it has 10 classes, rerun, and watch the same call succeed")

change y11 so it has 10 classes, rerun, and watch the same call succeed


## Solutions

**Your turn 2, a different split.** Set `SEED = 7` in the split cell and rerun the three cells after it. On the episode machine every seed still landed at 1.0000, wine is that easy, which is exactly why section 3 leaves wine.

```python
SEED = 7   # then rerun split, fit + predict, accuracy
```

**Your turn 3, your own CSV.**

```python
from google.colab import files
uploaded = files.upload()                       # pick my_table.csv
!{sys.executable} benchmark.py my_table.csv --target label
```

If your label has more than 10 classes the script refuses with the reason before anything downloads, benchmark without TabFM via `--models xgboost,tabicl`.

**Your turn 4, ten classes fit.**

```python
y10 = np.arange(220) % 10          # exactly at the cap
clf10 = TabFMClassifier(model=model)
clf10.fit(X11, y10)                # goes through
print(clf10.predict(X11[:5]).astype(int))
```

## Take it home

Your coding agent can know everything this notebook taught, the verified setup, every sharp edge, the benchmark utility, and the Vertex AI GPU provisioning, in one command.

```
npx skills add https://github.com/SaschaHeyer/gen-ai-livestream/tree/main/tabfm/skill
```

The full written deep dive, [ARTICLE LINK]

If this saved you the confusing hours it cost me, I make one of these every week, [youtube.com/@ml-engineer](https://www.youtube.com/@ml-engineer).